# 00 — Bootstrap do Colab (rodar a cada sessão)

Reconstrói o ambiente do zero: **GPU → Drive → clona frameworks → instala deps fixadas → reinicia runtime → sanity check → sincroniza dados**.

> Abra este notebook via **File → Open notebook → GitHub** e cole a URL deste repositório.

**Antes de rodar:** ajuste `PROJECT_REPO_URL` abaixo para o SEU fork/repo.

In [ ]:
# --- Configuração (EDITE AQUI) --------------------------------------------
PROJECT_REPO_URL = "https://github.com/SEU_USUARIO/tcc-guitar.git"  # <-- troque
PROJECT_DIR      = "/content/tcc-guitar"

# Frameworks (clonados em runtime). Fixe um commit p/ reprodutibilidade se quiser.
FRAMEWORKS = {
    "amt-tools": "https://github.com/cwitkowitz/amt-tools.git",
    "guitar-transcription-with-inhibition":
        "https://github.com/cwitkowitz/guitar-transcription-with-inhibition.git",
    "guitar-transcription-continuous":
        "https://github.com/cwitkowitz/guitar-transcription-continuous.git",
}

# Onde os dados persistentes vivem no seu Drive (usado por src/paths.py).
import os
os.environ["TCC_DRIVE_ROOT"] = "/content/drive/MyDrive/tcc-guitar"
os.environ["TCC_LOCAL_ROOT"] = "/content/tcc-data"


## 1. Checar GPU

In [ ]:
!nvidia-smi

## 2. Montar o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clonar o projeto e os frameworks

In [ ]:
import os, subprocess

def clone(url, dst):
    if os.path.isdir(dst):
        print(f"[skip] {dst} já existe")
        return
    print(f"[clone] {url} -> {dst}")
    subprocess.run(["git", "clone", "--depth", "1", url, dst], check=True)

# Projeto
clone(PROJECT_REPO_URL, PROJECT_DIR)

# Frameworks (em /content)
for name, url in FRAMEWORKS.items():
    clone(url, f"/content/{name}")

os.chdir(PROJECT_DIR)
print("cwd:", os.getcwd())

## 4. Instalar dependências FIXADAS, depois os frameworks

Ordem importa: primeiro fixamos a constelação frágil (numpy/librosa/jams/numba),
depois instalamos os frameworks com `-e`. Como os repos usam `>=`, o pip **não**
faz upgrade do que já satisfaz a restrição, então os pins seguram.

In [ ]:
# 4a. pins frágeis
!pip install -q -r /content/tcc-guitar/requirements-colab.txt

# 4b. frameworks em modo editável (respeitam os pins já instalados)
!pip install -q -e /content/amt-tools
!pip install -q -e /content/guitar-transcription-with-inhibition
!pip install -q -e /content/guitar-transcription-continuous
print("Instalação concluída.")

## 5. ⚠️ REINICIAR O RUNTIME

O `numpy` foi rebaixado (Colab vem com 2.x; fixamos 1.23.5). O kernel precisa
reiniciar para carregar a versão nova. **Rode a célula abaixo** — ela reinicia
sozinha. Depois, **pule as células 1–5 e continue da célula 6.**

In [ ]:
import os
os.kill(os.getpid(), 9)  # força restart do runtime do Colab

## 6. Sanity check do ambiente (portão da Fase 0)

Reexecute a célula de **Configuração** (topo) antes desta, pois o restart limpou
as variáveis. Depois rode o check.

In [ ]:
import os, sys
# Reconfigurar após o restart:
os.environ.setdefault("TCC_DRIVE_ROOT", "/content/drive/MyDrive/tcc-guitar")
os.environ.setdefault("TCC_LOCAL_ROOT", "/content/tcc-data")
os.chdir("/content/tcc-guitar")
sys.path.insert(0, "/content/tcc-guitar")

!python -m src.env_check

## 7. Conferir os caminhos e criar as pastas no Drive

In [ ]:
from src.paths import P
P.print_summary()
P.ensure_dirs()
print("\npastas garantidas no Drive.")

## 8. Sincronizar dados (Drive → disco local)

Use conforme a fase. Na 1ª vez (antes de ter cache), você vai baixar o GuitarSet
bruto para `raw/guitarset` no Drive e processar (Fase 0). Depois que o cache
existir empacotado no Drive, esta célula reconstrói a bancada em segundos.

In [ ]:
from src import data_sync

# (a) trazer áudio bruto do Drive p/ local (quando já estiver no Drive):
# data_sync.rsync_raw("guitarset")

# (b) trazer um cache já processado (depois da Fase 0):
# data_sync.pull_archive("gset_cache", P.local_root)

print("Pronto. Ajuste as chamadas acima conforme a fase.")

---
### Plano B — `condacolab` (só se a Fase 0 brigar com jams/numba)

Se `env_check` insistir em falhar por incompatibilidade de Python, troque a
instalação por um ambiente conda com Python 3.10:

```python
!pip install -q condacolab
import condacolab; condacolab.install()   # reinicia o runtime sozinho
# depois: !conda create -n tcc python=3.10 -y  ... e instalar dentro dele
```
Deixe isto como último recurso — tentar os pins nativos primeiro é mais simples.